# Lab - A Regression Gate for Prompt Changes

Three tasks on measuring a prompt change instead of guessing. Every edit changes some answers for the better and some for the worse - a test suite with results per case, and a gate before release, tells you which.

| Task | What you learn |
| --- | --- |
| 1 | Read the verdict defensively, and keep one result per case |
| 2 | Read the failing cases, not just the score |
| 3 | A gate that blocks a change when any passing case breaks |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# The test set: a customer question and the verdict we expect.
CASES = [
    ("I bought a jacket, delivered 20 days ago. Can I return it?", "YES"),
    ("The shoes arrived yesterday. I want to send them back.", "YES"),
    ("My order was delivered 45 days ago. Can I still return it?", "NO"),
    ("I got a gift card last week. Can I return it?", "NO"),
    ("The underwear I ordered came 3 days ago. Return possible?", "NO"),
    ("Delivered 29 days ago, is a return still possible?", "YES"),
]

PROMPT_V1 = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Begin your answer with YES or NO, then one sentence."
)


def answer(system, text):
    response = client.messages.create(
        model=MODEL,
        max_tokens=100,
        output_config={"effort": "low"},
        system=system,
        messages=[{"role": "user", "content": text}],
    )
    for block in response.content:
        if block.type == "text":
            return block.text

---

## Task 1 - Results per case, not just a number

In [ ]:
# ============================================================
# TASK 1 - Results per case, not just a number
# ============================================================
#
# WHAT TO DO
#   Finish verdict(): read YES or NO from the start of a reply,
#   or None when neither is there. Then finish run_suite(): one
#   record per case, with the expected verdict, the verdict
#   read from the reply, and whether they match.
#
# WHY IT MATTERS
#   The prompt asks Claude to begin with YES or NO so that a
#   program can read the verdict - but a program has to read it
#   carefully: "Yes," and "YES." are the same answer, "Maybe" is
#   none. And a test run should keep every case's result, not
#   just a total: results per case can be compared, stored, and
#   diffed between two prompts.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the two functions".
#
# HOW TO DO IT
#   Three blanks:
#
#       ("YES", "NO")   the only verdicts a reply may start with
#       expected        the verdict the test set expects
#       got             the verdict read from the reply
# ============================================================

# TODO: replace each ... below with the value named beside it
def verdict(reply):
    words = (reply or "").strip().split()
    first = words[0].strip(".,:;!").upper() if words else ""
    return first if first in ... else None   # ("YES", "NO")


def run_suite(system):
    results = []
    for text, expected in CASES:
        got = verdict(answer(system, text))
        results.append({"text": text, "expected": ..., "got": got,   # expected
                        "pass": got == ...})   # expected
    return results

try:
    v1 = run_suite(PROMPT_V1)
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for r in v1:
    print(r["expected"], r["got"], "PASS" if r["pass"] else "FAIL", "|", r["text"][:50])

check("suite_results", verdict=verdict, v1=v1)

---

## Task 2 - Read the failures, not just the score

In [ ]:
# ============================================================
# TASK 2 - Read the failures, not just the score
# ============================================================
#
# WHAT TO DO
#   Write score() - how many cases passed - and pull the
#   failing cases out of the first prompt's results.
#
# WHY IT MATTERS
#   The score is below the maximum, so something is wrong. The
#   failing cases say what: Claude said yes to underwear,
#   because the prompt never mentions it. The prompt has a gap,
#   and the test found it in seconds. A score points at a
#   problem; the failures name it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The score is below the maximum, so let's look at the
#   fails".
#
# HOW TO DO IT
#   Three blanks:
#
#       "pass"         the field that says whether a case passed
#       False          keep the cases that did not pass
#       failures       the list to print
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def score(results):
        return sum(1 for r in results if r[...])   # "pass"
    
    failures = [r for r in v1 if r["pass"] is ...]   # False
    print("v1 score:", score(v1), "/", len(CASES))
    for r in ...:   # failures
        print("FAIL:", r["text"], "- expected", r["expected"], "got", r["got"])
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("read_failures", v1=v1, failures=failures, score=score)

---

## Task 3 - A regression gate before release

In [ ]:
# ============================================================
# TASK 3 - A regression gate before release
# ============================================================
#
# WHAT TO DO
#   Finish gate(): a new prompt may replace the old one only if
#   no case that passed before fails now, and the score did not
#   drop. Run it on two candidates: v2, the lecture's fix, and
#   v3, which fixes underwear but also shortens the window.
#
# WHY IT MATTERS
#   Every prompt edit changes some answers for the better and
#   some for the worse. Comparing totals is not enough: v3
#   scores the same as v1 while a customer at 29 days now gets
#   the wrong answer. A gate that compares case by case, run
#   before every release, catches that - the same cases, two
#   prompts, nothing allowed to get worse.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So the practical rule: keep a test set with expected
#   answers".
#
# HOW TO DO IT
#   Three blanks:
#
#       old["pass"]   the case passed with the old prompt...
#       new["pass"]   ...and fails with the new one
#       score(old)    the new score must not be lower than this
# ============================================================

PROMPT_V2 = PROMPT_V1 + " Gift cards and underwear cannot be returned."
PROMPT_V3 = PROMPT_V2.replace("within 30 days", "within 25 days")   # a careless edit

try:
    # TODO: replace each ... below with the value named beside it
    def gate(old_results, new_results):
        broken = [new["text"] for old, new in zip(old_results, new_results)
                  if ... and not ...]   # old["pass"], new["pass"]
        if broken:
            return False, "broke: " + "; ".join(broken)
        if score(new_results) < ...:   # score(old_results)
            return False, "the score dropped"
        return True, "no case got worse"

    v2 = run_suite(PROMPT_V2)
    v3 = run_suite(PROMPT_V3)
    for name, results in (("v2", v2), ("v3", v3)):
        ok, why = gate(v1, results)
        print(name, "score", score(results), "/", len(CASES), "->",
              "RELEASE" if ok else "BLOCKED", "-", why)
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("regression_gate", gate=gate, v1=v1, v2=v2, v3=v3)

---

## Done

Keep a test set with expected answers, keep the result of every case, and never ship a prompt change without a gate that compares case by case: the score may stay flat while a customer's answer gets worse.